In [1]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
import os

builder = SparkSession.builder \
    .appName("Test Delta") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [2]:
dim_path = "../data/dim/"
fact_path = "../data/fact/"
bronze_path = "../delta_lake/bronze/"

In [3]:
dim_files = ["dim_artists.csv", "dim_albums.csv", "dim_genres.csv", "dim_tracks.csv"]
fact_files = ["fact_tracks.csv"]

In [4]:
def read_csv_with_options(path):
    return spark.read \
        .option("header", True) \
        .option("multiLine", True) \
        .option("quote", '"') \
        .option("escape", '"') \
        .option("mode", "PERMISSIVE") \
        .csv(path)

def bronze_load_and_save(csv_path, delta_path):
    df = read_csv_with_options(csv_path)
    df.write.format("delta").mode("overwrite").save(delta_path)
    print(f"Guardado Delta en: {delta_path}")

os.makedirs(bronze_path, exist_ok=True)

In [5]:
# dim
for file in dim_files:
    csv_file = os.path.join(dim_path, file)
    delta_file = os.path.join(bronze_path, file.replace(".csv", ""))
    bronze_load_and_save(csv_file, delta_file)

# fact
for file in fact_files:
    csv_file = os.path.join(fact_path, file)
    delta_file = os.path.join(bronze_path, file.replace(".csv", ""))
    bronze_load_and_save(csv_file, delta_file)

Guardado Delta en: ../delta_lake/bronze/dim_artists
Guardado Delta en: ../delta_lake/bronze/dim_albums
Guardado Delta en: ../delta_lake/bronze/dim_genres
Guardado Delta en: ../delta_lake/bronze/dim_tracks
Guardado Delta en: ../delta_lake/bronze/fact_tracks


In [6]:
df_test = spark.read.format("delta").load(os.path.join("../delta_lake/bronze/", "dim_tracks"))
df_test.show(5)

+--------------------+--------------------+--------+--------+
|            track_id|          track_name|album_id|genre_id|
+--------------------+--------------------+--------+--------+
|5SuOikwiRyPMVoIQD...|              Comedy|       1|       1|
|4qPNDBW1i3p13qLCt...|    Ghost - Acoustic|       2|       1|
|1iJBSr7s7jYXzM8EG...|      To Begin Again|       3|       1|
|6lfxq3CG4xtTiEg7o...|Can't Help Fallin...|       4|       1|
|5vjLSffimiIP26QG5...|             Hold On|       5|       1|
+--------------------+--------------------+--------+--------+
only showing top 5 rows

